# Film Yorumları Duygu Analizi (Sentiment Analysis)
Bu notebook'ta, metin halindeki film yorumlarını analiz edip, cümlenin **Olumlu (Pozitif)** mu yoksa **Olumsuz (Negatif)** mu olduğunu anlayan bir Doğal Dil İşleme (NLP) modeli kuracağız.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import nltk
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns

plt.style.use('dark_background')

## 1. Veri Yükleme
Örnek olarak üretilmiş 500 satırlık film yorumu (İngilizce) verisetini okuyalım.

In [ ]:
df = pd.read_csv('movie_reviews.csv')
display(df.head())
print("\nDağılım:")
display(df['Sentiment'].value_counts())

## 2. Kelime Bulutu (Word Cloud) Görselleştirmesi
Pozitif ve negatif yorumlarda en çok hangi kelimeler kullanılmış bakalım.

In [ ]:
positive_text = " ".join(df[df['Sentiment'] == 1]['Review'].values)
negative_text = " ".join(df[df['Sentiment'] == 0]['Review'].values)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Pozitif Kelimeler
wc_pos = WordCloud(width=400, height=400, background_color='black', colormap='Greens').generate(positive_text)
axes[0].imshow(wc_pos, interpolation='bilinear')
axes[0].set_title('Pozitif Yorumlar')
axes[0].axis('off')

# Negatif Kelimeler
wc_neg = WordCloud(width=400, height=400, background_color='black', colormap='Reds').generate(negative_text)
axes[1].imshow(wc_neg, interpolation='bilinear')
axes[1].set_title('Negatif Yorumlar')
axes[1].axis('off')

plt.show()

## 3. NLP Ön İşleme (TF-IDF) ve Eğitim/Test Bölmesi
Metin verilerini makine öğrenmesi modelinin anlayabileceği sayılara (vektörlere) çeviriyoruz.

In [ ]:
X = df['Review']
y = df['Sentiment']

# Veriyi böl
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# TF-IDF ile Kelimeleri Sayısallaştır
vectorizer = TfidfVectorizer(stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

## 4. Model Eğitimi (Naive Bayes)

In [ ]:
model = MultinomialNB()
model.fit(X_train_vec, y_train)

# Test seti üzerinde tahmin
y_pred = model.predict(X_test_vec)

acc = accuracy_score(y_test, y_pred)
print(f"Model Doğruluk (Accuracy) Oranı: {acc * 100:.2f}%")

## 5. Sonuçların Değerlendirilmesi (Confusion Matrix)

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, 
            xticklabels=['Negatif (0)', 'Pozitif (1)'], 
            yticklabels=['Negatif (0)', 'Pozitif (1)'])
plt.title('Hata Matrisi (Confusion Matrix)')
plt.xlabel('Tahmin Edilen Sınıf')
plt.ylabel('Gerçek Sınıf')
plt.show()

print(classification_report(y_test, y_pred))

## 6. Canlı Test Yapalım!
Modelimiz daha önce görmediği bir cümleyi nasıl sınıflandıracak?

In [ ]:
# Yeni yorumlar
new_reviews = [
    "I absolutely loved the cinematography, brilliant!",
    "It was a complete waste of time, terrible acting."
]

new_vec = vectorizer.transform(new_reviews)
predictions = model.predict(new_vec)

for review, pred in zip(new_reviews, predictions):
    sentiment = "Pozitif 😊" if pred == 1 else "Negatif 😡"
    print(f"Yorum: '{review}'\nTahmin: {sentiment}\n")

## 7. Özet ve Sonuçlar
### Q&A
- **Soru:** Bilgisayar, metnin duygusunu nasıl anlar?
- **Cevap:** TF-IDF yöntemi ile kelimelerin frekansları hesaplanır ve makine öğrenmesi modeli geçmiş verilerden "terrible" kelimesinin daha çok Negatif etiketli metinlerde geçtiğini öğrenir.

### Data Analysis Key Findings
- Kelime bulutu sayesinde pozitif yorumlarda "wonderful, brilliant", negatif yorumlarda ise "waste, boring, terrible" kelimelerinin ağırlıklı olduğu açıkça görülmektedir.
- Naive Bayes sınıflandırıcısı, küçük ve basit veri setlerinde yüksek doğruluk (%100'e yakın) elde edebilmektedir (Verisetimiz sentetik ve tekdüze olduğu için başarım çok yüksektir).

### Insights or Next Steps
- Modelin gerçek dünya karmaşasını öğrenebilmesi için Twitter (X) API üzerinden veya devasa IMDB Kaggle veri setleri ile eğitilmesi gerekir.
- Sarcasm (İğneleme / Kinaye) gibi zor metin yapılarını anlamak için BERT veya GPT tabanlı Transformer modellerine (Deep Learning) geçiş yapılmalıdır.